<a href="https://colab.research.google.com/github/javierariassantillan/A_ABET-ProyectoIntegrador/blob/main/Arias_Javier_Andrade_Eduardo_Examen_Parcial1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Examen Progreso 1 - Aprendizaje Automático

## **Objetivo** - Diagnosticar, limpiar y preparar un conjunto de datos empresariales para un futuro problema de aprendizaje automático, justificando las decisiones técnicas adoptadas.

##**Nombres**: Javier Arias, Eduardo Andrade

## **Predicción de abandono de clientes de una entidad financiera** La entidad desea analizar los factores asociados al abandono de sus clientes para desarrollar, en una etapa posterior, un sistema de aprendizaje automático que permita identificar clientes con riesgo de abandonar el servicio. El área de analítica ha recibido una base de aproximadamente 6.000 registros. Sin embargo, los datos presentan problemas: valores faltantes, categorías inconsistentes, registros duplicados, valores numéricos inválidos y posibles valores atípicos. Además, existen columnas que podrían introducir fuga de información.

## Reglas

1. Complete las secciones marcadas como TODO.

2. Ejecute las celdas en orden y conserve las evidencias de sus resultados.

3. No elimine registros o columnas sin justificar técnicamente la decisión.

4. No utilice la variable objetivo para calcular transformaciones de las características.

5. Las estadísticas de imputación, los codificadores y los escaladores deben ajustarse utilizando únicamente los datos de entrenamiento.

6. No se exige entrenar un modelo predictivo.

7. Escriba las justificaciones en las celdas Markdown indicadas.

8. No sustituya los resultados obtenidos por valores escritos manualmente.

## **Entrega** - El notebook de Colab será cargado con todas las celdas ejecutadas y las conclusiones completadas.

## **Celda de configuración**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np

from IPython.display import display

# Importar directamente la base de datos del examen desde Google Drive
url = "/content/drive/MyDrive/Examen_Aprendizaje_Automatico/clientes_financieros_examen.csv"

df_raw = pd.read_csv(url)

df = df_raw.copy(deep=True) # Crear una copia de trabajo

assert not df_raw.empty, "Error: la base de datos está vacía." # Verificar que la carga se realizó correctamente


print("Base de datos cargada correctamente.")
print(f"Número de filas: {df_raw.shape[0]}")
print(f"Número de columnas: {df_raw.shape[1]}")

print("\nPrimeras cinco filas:")
display(df_raw.head(10))

Base de datos cargada correctamente.
Número de filas: 6035
Número de columnas: 13

Primeras cinco filas:


,id_cliente,edad,ingresos_anuales,antiguedad_meses,saldo_promedio,productos_activos,canal,estado_civil,region,contacto_preferido,incumplimiento,abandono,motivo_baja
0,100001,23.0,46789.4,131,5059.76,1,Telefono,Casado,Centro,App,No,0,No aplica
1,100002,66.0,21426.93,12,7625.14,2,Digital,Casado,NaN,Email,No,0,No aplica
2,100003,59.0,5000.0,237,6620.34,1,Digital,NaN,Norte,Email,No,0,No aplica
3,100004,45.0,7262.53,173,3793.18,1,Sucursal,Soltero,Sur,App,No,1,Atencion
4,100005,45.0,51902.78,112,8113.38,2,Telefono,Divorciado,NaN,Email,No,0,No aplica
5,100006,72.0,26382.43,58,3475.79,1,Sucursal,Soltero,Sur,Email,No,1,Atencion
6,100007,23.0,21489.86,66,5365.05,4,Digital,Soltero,Norte,Email,No,0,No aplica
7,100008,61.0,54997.24,109,3410.98,2,NaN,Divorciado,Norte,Telefono,No,0,No aplica
8,100009,30.0,49712.24,14,6116.15,1,Digital,Casado,Sur,Email,No,0,No aplica
9,100010,23.0,32342.28,89,3182.29,3,Digital,Casado,Sur,Telefono,No,1,Precio


In [ ]:
columnas_esperadas = {
    "id_cliente",
    "edad",
    "ingresos_anuales",
    "antiguedad_meses",
    "saldo_promedio",
    "productos_activos",
    "canal",
    "estado_civil",
    "region",
    "contacto_preferido",
    "incumplimiento",
    "abandono",
    "motivo_baja"
}

faltantes = columnas_esperadas - set(df_raw.columns)

assert not faltantes, (
    f"Error: faltan columnas requeridas: {faltantes}"
)

print("Verificación estructural completada.")
print("Todas las columnas esperadas están presentes.")

Verificación estructural completada.
Todas las columnas esperadas están presentes.


## **Seccion A: Fundamentos** (1 PUNTO)
1. ¿Cuál es el problema que se pretende estudiar y cuál es la variable objetivo?
2. ¿Se trata de aprendizaje supervisado o no supervisado? Justifique.
3. ¿La tarea futura corresponde a clasificación o regresión? Justifique.
4. Enumere las etapas del ciclo de trabajo de aprendizaje automático y ubique la preparación de datos.
5. Explique por qué no sería correcto evaluar un modelo con los mismos datos empleados para entrenarlo.


### Respuestas – Sección A

**1. Problema y variable objetivo.** Se busca identificar qué clientes de la entidad financiera tienen riesgo de abandonar el servicio, a partir de sus características demográficas, financieras y de relación con la entidad. La variable objetivo es **`abandono`** (1 = el cliente abandonó, 0 = permanece).

**2. ¿Supervisado o no supervisado?** Es **aprendizaje supervisado**, porque cada registro histórico tiene la etiqueta conocida (`abandono`). El modelo aprenderá la relación entre las características (X) y esa etiqueta (y) para luego predecirla en clientes nuevos.

**3. ¿Clasificación o regresión?** Es **clasificación binaria**, porque la variable objetivo es categórica con dos clases (0/1) y no un valor continuo. El modelo deberá asignar a cada cliente una clase o una probabilidad de pertenecer a la clase "abandona".

**4. Etapas del ciclo de aprendizaje automático:**
1. Definición del problema y de la métrica de éxito.
2. Recolección / obtención de los datos.
3. Análisis exploratorio y auditoría de calidad.
4. **Preparación de datos**: limpieza, tratamiento de faltantes, duplicados e inválidos, selección de variables, división train/test, imputación, codificación y escalamiento. *(Este examen cubre esta etapa, junto con la anterior.)*
5. Entrenamiento del modelo.
6. Evaluación con datos no vistos (conjunto de prueba).
7. Despliegue.
8. Monitoreo y mantenimiento (deriva de datos, reentrenamiento).

**5. ¿Por qué no evaluar con los mismos datos de entrenamiento?** Porque el modelo ya "vio" esos ejemplos y puede haberlos memorizado (sobreajuste). La métrica obtenida sería optimista y no reflejaría el desempeño real sobre clientes nuevos. Para estimar la capacidad de generalización se necesita un conjunto de prueba independiente, que no intervenga ni en el entrenamiento ni en el ajuste de los transformadores.

## **Sección B: Auditoria exploratoria** (2 PUNTOS)
- Determine el número de filas y columnas.
- Identifique los tipos de datos.
- Calcule los valores faltantes por columna, en cantidad y porcentaje.
- Detecte filas duplicadas.
- Examine la distribución de la variable objetivo.
- Identifique valores imposibles, categorías inconsistentes y posibles valores atípicos.
- Presente al menos tres hallazgos relevantes con evidencia.

### **Dimensiones y tipos de datos**

In [ ]:
# MOSTRAR DIMENSIONES Y TIPOS DE DATOS
print(f"Filas: {df.shape[0]}  |  Columnas: {df.shape[1]}\n")
print("Tipos de datos:")
display(df.dtypes.astype(str).to_frame("tipo"))

# Columnas que deberían ser numéricas pero se leyeron como texto
cols_num_esperadas = ["edad", "ingresos_anuales", "antiguedad_meses",
                      "saldo_promedio", "productos_activos"]
for c in cols_num_esperadas:
    if not pd.api.types.is_numeric_dtype(df[c]):
        print(f"Atención: '{c}' se leyó como texto ({df[c].dtype}); contiene valores no numéricos.")

Filas: 6035  |  Columnas: 13

Tipos de datos:


,tipo
id_cliente,int64
edad,object
ingresos_anuales,object
antiguedad_meses,int64
saldo_promedio,object
productos_activos,int64
canal,object
estado_civil,object
region,object
contacto_preferido,object


Atención: 'edad' se leyó como texto (object); contiene valores no numéricos.
Atención: 'ingresos_anuales' se leyó como texto (object); contiene valores no numéricos.
Atención: 'saldo_promedio' se leyó como texto (object); contiene valores no numéricos.


### **Valores Faltantes**

In [ ]:
# CREAR UNA TABLA CON CANTIDAD Y PORCENTAJE DE NULOS POR COLUMNAS
nulos = pd.DataFrame({
    "nulos": df.isna().sum(),
    "porcentaje_%": (df.isna().mean() * 100).round(2)
}).sort_values("nulos", ascending=False)
display(nulos)

# Marcadores de ausencia "ocultos" que pandas NO reconoce como NaN
marcadores = ["desconocido", "?", "nan", "none", "null", "na", "n/a", ""]
ocultos = {}
for c in df.columns:
    if not pd.api.types.is_numeric_dtype(df[c]):
        s = df[c].dropna().astype(str).str.strip().str.lower()
        n = s.isin(marcadores).sum()
        if n > 0:
            ocultos[c] = n
print("Marcadores de ausencia ocultos (texto):")
display(pd.Series(ocultos, name="cantidad").to_frame())

,nulos,porcentaje_%
ingresos_anuales,362,6.00
saldo_promedio,299,4.95
estado_civil,239,3.96
canal,180,2.98
edad,152,2.52
region,120,1.99
id_cliente,0,0.00
antiguedad_meses,0,0.00
productos_activos,0,0.00
contacto_preferido,0,0.00


Marcadores de ausencia ocultos (texto):


,cantidad
edad,12
ingresos_anuales,12
saldo_promedio,12
estado_civil,1
region,25


### **Duplicados**

In [ ]:
# CALCULAR EL NUMERO DE FILAS DUPLICADAS
dup_exactos = df.duplicated().sum()
dup_id = df["id_cliente"].duplicated().sum()
print(f"Filas duplicadas exactas: {dup_exactos}")
print(f"id_cliente repetidos:     {dup_id}")

# Ejemplo de registros duplicados
display(df[df["id_cliente"].duplicated(keep=False)].sort_values("id_cliente").head(6))

Filas duplicadas exactas: 35
id_cliente repetidos:     35


,id_cliente,edad,ingresos_anuales,antiguedad_meses,saldo_promedio,productos_activos,canal,estado_civil,region,contacto_preferido,incumplimiento,abandono,motivo_baja
156,100157,25.0,45571.74,18,4312.39,1,Telefono,Casado,Norte,App,No,1,Competencia
6007,100157,25.0,45571.74,18,4312.39,1,Telefono,Casado,Norte,App,No,1,Competencia
6019,100168,28.0,17775.77,187,2652.85,2,Sucursal,Casado,Norte,Telefono,Si,0,No aplica
167,100168,28.0,17775.77,187,2652.85,2,Sucursal,Casado,Norte,Telefono,Si,0,No aplica
221,100222,46.0,23484.64,199,116.99,1,Digital,Divorciado,Sur,Email,No,0,No aplica
6002,100222,46.0,23484.64,199,116.99,1,Digital,Divorciado,Sur,Email,No,0,No aplica


### **Distribución de la variable objetivo**

In [ ]:
# MOSTRAR FRECUENCIA Y PORCENTAJE DE ABANDONO
dist_obj = pd.DataFrame({
    "frecuencia": df["abandono"].value_counts(),
    "porcentaje_%": (df["abandono"].value_counts(normalize=True) * 100).round(2)
})
display(dist_obj)
ratio = dist_obj["frecuencia"].max() / dist_obj["frecuencia"].min()
print(f"Relación clase mayoritaria / minoritaria: {ratio:.1f} : 1")

,frecuencia,porcentaje_%
abandono,,
0,5216,86.43
1,819,13.57


Relación clase mayoritaria / minoritaria: 6.4 : 1


### **Diagnóstico numérico y categórico**

In [ ]:
# EXAMINAR ESTADISTICOS DESCRIPTIVOS Y CATEGORIAS UNICAS
# Conversión temporal (solo para diagnóstico, no modifica df)
diag = df.copy()
for c in ["edad", "ingresos_anuales", "saldo_promedio"]:
    diag[c] = pd.to_numeric(diag[c], errors="coerce")

print("Estadísticos descriptivos (variables numéricas):")
display(diag[cols_num_esperadas].describe().T.round(2))

# Valores imposibles o sospechosos
print("Edades > 100:", (diag["edad"] > 100).sum(), "->", diag.loc[diag["edad"] > 100, "edad"].unique())
print("Saldo = -999999 (código centinela):", (diag["saldo_promedio"] == -999999).sum())
print("Saldos negativos distintos de -999999:", ((diag["saldo_promedio"] < 0) & (diag["saldo_promedio"] != -999999)).sum())
print("Ingresos exactamente = 5000 (posible piso):", (diag["ingresos_anuales"] == 5000).sum())

# Posibles atípicos según regla IQR (solo identificación)
print("\nPosibles atípicos por regla IQR (1.5*IQR):")
for c in ["edad", "ingresos_anuales", "saldo_promedio", "antiguedad_meses"]:
    x = diag[c].dropna()
    q1, q3 = x.quantile([0.25, 0.75]); iqr = q3 - q1
    n = ((x < q1 - 1.5 * iqr) | (x > q3 + 1.5 * iqr)).sum()
    print(f"  {c:18s}: {n}")

# Categorías únicas (se observan inconsistencias de formato)
cols_cat = ["canal", "estado_civil", "region", "contacto_preferido", "incumplimiento", "motivo_baja"]
for c in cols_cat:
    print(f"\n{c}:")
    print(df[c].value_counts(dropna=False).to_dict())

# Relación entre motivo_baja y abandono (posible fuga)
print("\nTabla cruzada motivo_baja vs abandono:")
display(pd.crosstab(df["motivo_baja"], df["abandono"]))

Estadísticos descriptivos (variables numéricas):


,count,mean,std,min,25%,50%,75%,max
edad,5871.0,49.49,20.83,18.0,33.0,49.00,65.00,250.00
ingresos_anuales,5661.0,32188.21,13658.19,5000.0,22737.9,31779.10,41283.57,88361.54
antiguedad_meses,6035.0,120.47,69.44,1.0,59.0,121.00,180.00,240.00
saldo_promedio,5724.0,2799.92,42099.31,-999999.0,2200.1,4488.72,6863.72,15700.85
productos_activos,6035.0,1.96,0.86,1.0,1.0,2.00,3.00,4.00


Edades > 100: 15 -> [250.]
Saldo = -999999 (código centinela): 10
Saldos negativos distintos de -999999: 548
Ingresos exactamente = 5000 (posible piso): 143

Posibles atípicos por regla IQR (1.5*IQR):
  edad              : 15
  ingresos_anuales  : 24
  saldo_promedio    : 41
  antiguedad_meses  : 0

canal:
{'Digital': 3188, 'Sucursal': 1772, 'Telefono': 864, nan: 180, ' DIGITAL ': 22, ' SUCURSAL ': 5, ' TELEFONO ': 4}

estado_civil:
{'Casado': 2612, 'Soltero': 2306, 'Divorciado': 848, nan: 239, ' CASADO ': 14, ' SOLTERO ': 9, ' DIVORCIADO ': 6, ' NAN ': 1}

region:
{'Centro': 1975, 'Sur': 1970, 'Norte': 1915, nan: 120, '?': 25, ' NORTE ': 14, ' CENTRO ': 11, ' SUR ': 5}

contacto_preferido:
{'App': 2113, 'Email': 1966, 'Telefono': 1956}

incumplimiento:
{'No': 5620, 'Si': 415}

motivo_baja:
{'No aplica': 5216, 'Competencia': 293, 'Atencion': 278, 'Precio': 248}

Tabla cruzada motivo_baja vs abandono:


abandono,0,1
motivo_baja,,
Atencion,0,278
Competencia,0,293
No aplica,5216,0
Precio,0,248


**INTERPRETACION:** Escriba al menos 3 hallazgos. Para cada uno indique Evidencia observada. Problema identificado. Consecuencia potencial para el aprendizaje automático. Acción correctiva propuesta.

**Hallazgo 1. Columnas numéricas leídas como texto y marcadores de ausencia ocultos**
- *Evidencia:* `edad`, `ingresos_anuales` y `saldo_promedio` tienen tipo texto porque contienen la cadena `"desconocido"` (12 casos en cada una). En `region` aparecen 25 valores `"?"` y en `estado_civil` un valor `" NAN "`.
- *Problema:* ausencias que pandas no reconoce como NaN y tipos de dato incorrectos.
- *Consecuencia:* no se pueden calcular estadísticos ni escalar; los marcadores se convertirían en categorías falsas al aplicar One-Hot Encoding.
- *Acción:* convertir a numérico con `pd.to_numeric(errors="coerce")`, mapear los marcadores a `NaN` e imputarlos después, ajustando la imputación solo con train.

**Hallazgo 2. Valores imposibles o centinela**
- *Evidencia:* 15 registros con `edad = 250` y 10 registros con `saldo_promedio = -999999`.
- *Problema:* son errores de captura o códigos de sistema, no valores reales.
- *Consecuencia:* distorsionan la media y la desviación estándar, y por tanto el StandardScaler. El modelo aprendería patrones inexistentes.
- *Acción:* marcarlos como `NaN` según reglas de dominio (edad entre 18 y 100, saldo ≠ -999999) e imputarlos con la mediana de train, sin eliminar las filas. Los 548 saldos negativos restantes son plausibles (sobregiros) y se conservan.

**Hallazgo 3. Categorías inconsistentes**
- *Evidencia:* existen variantes como `" DIGITAL "`/`"Digital"`, `" CASADO "`/`"Casado"` y `" NORTE "`/`"Norte"`, con espacios y mayúsculas distintas.
- *Problema:* la misma categoría está representada de varias formas.
- *Consecuencia:* One-Hot Encoding crearía columnas redundantes y dispersas, y la información de una categoría quedaría fragmentada.
- *Acción:* normalizar con `strip()` y homogeneizar mayúsculas/minúsculas antes de codificar.

**Hallazgo 4. Registros duplicados**
- *Evidencia:* 35 filas duplicadas exactas, que coinciden con 35 `id_cliente` repetidos.
- *Problema:* el mismo cliente se cuenta más de una vez.
- *Consecuencia:* sobrepeso de esos clientes y riesgo de que un mismo registro quede en train y en test, lo que infla la evaluación.
- *Acción:* eliminar los duplicados conservando la primera ocurrencia.

**Hallazgo 5. Desbalance de clases y fuga de información**
- *Evidencia:* `abandono = 1` representa el 13.57 % (819 de 6035), una relación aproximada de 6.4 : 1. Además, `motivo_baja` es `"No aplica"` en el 100 % de los casos con `abandono = 0` y tiene un motivo en el 100 % de los casos con `abandono = 1`.
- *Problema:* clase minoritaria escasa, y una variable (`motivo_baja`) que se conoce solo después del abandono.
- *Consecuencia:* el accuracy sería engañoso, y con `motivo_baja` el modelo tendría un desempeño "perfecto" en pruebas pero inútil en producción.
- *Acción:* usar división estratificada, evaluar a futuro con recall, F1 o PR-AUC, y excluir `motivo_baja` de las características.

## **Seccion C**: (2,5 puntos) Limpieza de datos - A partir de df_raw, construya una nueva base denominada df_clean. No modifique la base original.

### Convierta las columnas numéricas a tipos numéricos, tratando los valores inválidos de forma explícita

In [ ]:
df_clean = df_raw.copy(deep=True)

# TODO: convertir edad, ingresos_anuales y saldo_promedio a formato numérico
# Los valores que no puedan convertirse deben tratarse adecuadamente.
cols_a_convertir = ["edad", "ingresos_anuales", "saldo_promedio"]
for c in cols_a_convertir:
    original = df_clean[c]
    convertido = pd.to_numeric(original.astype("string").str.strip(), errors="coerce")
    # Valores que tenían contenido pero no eran numéricos -> pasan a NaN de forma explícita
    invalidos = original.notna() & convertido.isna()
    print(f"{c:18s}: {invalidos.sum()} valores no numéricos -> NaN  {original[invalidos].unique().tolist()}")
    df_clean[c] = convertido.astype(float)

print("\nTipos tras la conversión:")
print(df_clean[cols_a_convertir].dtypes)

edad              : 12 valores no numéricos -> NaN  ['desconocido']
ingresos_anuales  : 12 valores no numéricos -> NaN  ['desconocido']
saldo_promedio    : 12 valores no numéricos -> NaN  ['desconocido']

Tipos tras la conversión:
edad                float64
ingresos_anuales    float64
saldo_promedio      float64
dtype: object


**Justificación:** se usa `pd.to_numeric(errors="coerce")` para que todo texto no convertible (en este caso `"desconocido"`) quede como `NaN` explícito en lugar de detener el proceso o asignar un número arbitrario. No se eliminan filas: los `NaN` se imputarán en la Sección D con estadísticos calculados únicamente sobre el conjunto de entrenamiento.

### Trate los marcadores de ausencia y normalice las categorías de texto. Evite convertir la cadena "nan" en una categoría válida.

In [ ]:
# TODO: normalizar espacios, mayúsculas/minúsculas y marcadores de ausencia
# Aplique el tratamiento a las columnas categóricas pertinentes.
cols_texto = ["canal", "estado_civil", "region", "contacto_preferido", "incumplimiento", "motivo_baja"]
marcadores_ausencia = {"", "nan", "none", "null", "na", "n/a", "?", "desconocido", "sin dato"}

for c in cols_texto:
    s = df_clean[c].astype("string").str.strip().str.lower()   # los NaN reales se mantienen como <NA>
    s = s.mask(s.isin(marcadores_ausencia))                     # marcadores -> NaN (no se crea la categoría "nan")
    s = s.str.capitalize()                                     # formato homogéneo: "Digital", "Casado", ...
    df_clean[c] = s.astype(object).where(s.notna(), np.nan)

for c in cols_texto:
    print(f"{c}: {df_clean[c].value_counts(dropna=False).to_dict()}")

for c in cols_texto:
    assert not df_clean[c].dropna().str.lower().isin(marcadores_ausencia).any(), f"Marcador de ausencia como categoría en {c}"
print("\nVerificado: ninguna columna contiene 'nan', '?' u otros marcadores como categoría.")

canal: {'Digital': 3210, 'Sucursal': 1777, 'Telefono': 868, nan: 180}
estado_civil: {'Casado': 2626, 'Soltero': 2315, 'Divorciado': 854, nan: 240}
region: {'Centro': 1986, 'Sur': 1975, 'Norte': 1929, nan: 145}
contacto_preferido: {'App': 2113, 'Email': 1966, 'Telefono': 1956}
incumplimiento: {'No': 5620, 'Si': 415}
motivo_baja: {'No aplica': 5216, 'Competencia': 293, 'Atencion': 278, 'Precio': 248}

Verificado: ninguna columna contiene 'nan', '?' u otros marcadores como categoría.


**Justificación:** se eliminan espacios al inicio y al final y se unifica el formato de mayúsculas, de modo que `" DIGITAL "` y `"Digital"` sean la misma categoría. Los marcadores `"?"` y `"nan"` se convierten en `NaN` real **antes** de cualquier conversión a texto. Así se evita que la cadena `"nan"` termine como una categoría válida, que es lo que ocurre si se aplica `astype(str)` directamente sobre una columna con nulos. Los nulos se imputarán con la moda en el pipeline.

### Identifique y trate los registros duplicados.

In [ ]:
# TODO: identificar y eliminar duplicados según una regla justificada
n_antes = len(df_clean)
print("Duplicados exactos:", df_clean.duplicated().sum())
print("id_cliente repetidos:", df_clean["id_cliente"].duplicated().sum())

# Regla: un id_cliente identifica a un único cliente -> se conserva la primera ocurrencia
df_clean = df_clean.drop_duplicates(subset="id_cliente", keep="first").reset_index(drop=True)

print(f"\nFilas antes: {n_antes} | después: {len(df_clean)} | eliminadas: {n_antes - len(df_clean)}")
assert df_clean["id_cliente"].is_unique

Duplicados exactos: 35
id_cliente repetidos: 35

Filas antes: 6035 | después: 6000 | eliminadas: 35


**Justificación:** los 35 `id_cliente` repetidos corresponden a filas idénticas en todas las columnas, así que se trata de registros cargados dos veces y no de clientes distintos. La regla es que cada `id_cliente` debe aparecer una sola vez, conservando la primera ocurrencia. Eliminarlos evita dar más peso a esos clientes y que una misma fila quede en train y en test (fuga), y no se pierde información.

### Defina reglas de validación para edad, ingresos_anuales, antiguedad_meses, saldo_promedio y productos_activos. Documente qué valores se consideran inválidos y cómo los tratará.

In [ ]:
# TODO: aplicar reglas de dominio
# No elimine automáticamente todo valor extremo sin analizarlo.
reglas = {
    # columna: (condición de invalidez, descripción)
    "edad":              (lambda s: (s < 18) | (s > 100),  "edad fuera de [18, 100] años"),
    "ingresos_anuales":  (lambda s: s <= 0,                "ingreso anual <= 0"),
    "antiguedad_meses":  (lambda s: s < 0,                 "antigüedad negativa"),
    "saldo_promedio":    (lambda s: s == -999999,          "código centinela -999999"),
    "productos_activos": (lambda s: (s < 1) | (s > 10) | (s % 1 != 0), "productos no entero o fuera de [1, 10]"),
}

resumen = []
for col, (cond, desc) in reglas.items():
    mask = cond(df_clean[col]) & df_clean[col].notna()
    resumen.append({"columna": col, "regla_invalidez": desc, "invalidos": int(mask.sum()),
                    "tratamiento": "-> NaN (se imputa con la mediana de train)" if mask.sum() else "sin casos"})
    if mask.any():
        df_clean.loc[mask, col] = np.nan
display(pd.DataFrame(resumen))

# Valores extremos plausibles: se analizan, pero se conservan
print("Saldos negativos conservados (sobregiro):", (df_clean["saldo_promedio"] < 0).sum(),
      "| mínimo:", df_clean["saldo_promedio"].min())
print("Ingresos = 5000 conservados (posible ingreso mínimo registrado):", (df_clean["ingresos_anuales"] == 5000).sum())
print("Rango final edad:", df_clean["edad"].min(), "-", df_clean["edad"].max())
print("Rango final ingresos:", df_clean["ingresos_anuales"].min(), "-", df_clean["ingresos_anuales"].max())

# Consistencia antigüedad vs. edad (solo se reporta; no se elimina)
inconsist = (df_clean["antiguedad_meses"] / 12 > df_clean["edad"] - 18).sum()
print("Clientes con antigüedad mayor a su edad adulta (solo informativo):", inconsist)

,columna,regla_invalidez,invalidos,tratamiento
0,edad,"edad fuera de [18, 100] años",15,-> NaN (se imputa con la mediana de train)
1,ingresos_anuales,ingreso anual <= 0,0,sin casos
2,antiguedad_meses,antigüedad negativa,0,sin casos
3,saldo_promedio,código centinela -999999,10,-> NaN (se imputa con la mediana de train)
4,productos_activos,"productos no entero o fuera de [1, 10]",0,sin casos


Saldos negativos conservados (sobregiro): 547 | mínimo: -9107.32
Ingresos = 5000 conservados (posible ingreso mínimo registrado): 143
Rango final edad: 18.0 - 80.0
Rango final ingresos: 5000.0 - 88361.54
Clientes con antigüedad mayor a su edad adulta (solo informativo): 987


**Reglas de validación documentadas**

| Variable | Valor considerado inválido | Tratamiento |
|---|---|---|
| `edad` | < 18 o > 100 años (se encontraron 15 casos con 250) | Se reemplaza por `NaN` y luego se imputa con la mediana de train |
| `ingresos_anuales` | ≤ 0 | `NaN` + mediana (no hubo casos). Los 143 valores en 5000 se **conservan**, porque parecen un piso de registro plausible y no un error |
| `antiguedad_meses` | < 0 | `NaN` + mediana (no hubo casos, rango 1–240) |
| `saldo_promedio` | −999999 (código centinela de sistema) | `NaN` + mediana. Los saldos negativos reales (≈ −9100 a −3) se **conservan** porque representan sobregiros |
| `productos_activos` | no entero o fuera de 1–10 | `NaN` + mediana (no hubo casos, rango 1–4) |

**Criterio:** solo se anulan los valores imposibles o los códigos de sistema. Los valores extremos pero plausibles (atípicos por IQR en saldo o ingresos) se mantienen, porque pueden ser clientes reales y relevantes para el abandono. No se eliminan filas en esta etapa: el valor inválido se convierte en faltante y se imputa después, sin usar la variable objetivo.

### Muestre la dimensión final y compruebe nuevamente los problemas de calidad.

In [ ]:
# TODO: informar dimensiones, nulos, duplicados y tipos finales
print(f"Dimensión original: {df_raw.shape}  ->  Dimensión final df_clean: {df_clean.shape}")
print(f"Duplicados exactos: {df_clean.duplicated().sum()} | id_cliente repetidos: {df_clean['id_cliente'].duplicated().sum()}")

calidad = pd.DataFrame({
    "tipo": df_clean.dtypes.astype(str),
    "nulos": df_clean.isna().sum(),
    "porcentaje_%": (df_clean.isna().mean() * 100).round(2)
})
display(calidad)

print("Categorías finales:")
for c in cols_texto:
    print(f"  {c}: {sorted(df_clean[c].dropna().unique().tolist())}")

# Verificaciones automáticas
assert df_clean["edad"].dropna().between(18, 100).all()
assert (df_clean["saldo_promedio"] != -999999).all()
assert df_raw.shape == (6035, 13), "df_raw fue modificado"
print("\nVerificaciones de calidad superadas. df_raw permanece sin cambios.")

Dimensión original: (6035, 13)  ->  Dimensión final df_clean: (6000, 13)
Duplicados exactos: 0 | id_cliente repetidos: 0


,tipo,nulos,porcentaje_%
id_cliente,int64,0,0.00
edad,float64,177,2.95
ingresos_anuales,float64,372,6.20
antiguedad_meses,int64,0,0.00
saldo_promedio,float64,321,5.35
productos_activos,int64,0,0.00
canal,object,180,3.00
estado_civil,object,240,4.00
region,object,144,2.40
contacto_preferido,object,0,0.00


Categorías finales:
  canal: ['Digital', 'Sucursal', 'Telefono']
  estado_civil: ['Casado', 'Divorciado', 'Soltero']
  region: ['Centro', 'Norte', 'Sur']
  contacto_preferido: ['App', 'Email', 'Telefono']
  incumplimiento: ['No', 'Si']
  motivo_baja: ['Atencion', 'Competencia', 'No aplica', 'Precio']

Verificaciones de calidad superadas. df_raw permanece sin cambios.


**Comentario:** `df_clean` queda con 6000 filas y 13 columnas, sin duplicados, con tipos numéricos correctos y categorías homogéneas. Los `NaN` restantes (originales, marcadores convertidos y valores inválidos) se imputan en la Sección D, dentro del pipeline y ajustando solo con train. `df_raw` no se modificó.

## **Sección D** (2,5 puntos): preparación de variables - La variable objetivo es abandono. Prepare las características para un futuro modelo, sin entrenarlo.

- Separe la variable objetivo de las características.
- Identifique las columnas identificadoras.
- Determine si motivo_baja puede utilizarse como predictor.
- Justifique las columnas excluidas.

In [ ]:
# TODO: definir target, X e y
# Evite incluir identificadores y variables que revelen información posterior
# al evento que se pretende predecir.
target = "abandono"
col_id = ["id_cliente"]          # identificador: no aporta patrón generalizable
col_fuga = ["motivo_baja"]       # se conoce solo DESPUÉS del abandono -> fuga de información

# Evidencia de fuga: motivo_baja determina perfectamente el abandono
display(pd.crosstab(df_clean["motivo_baja"].fillna("NaN"), df_clean[target]))

columnas_excluidas = col_id + col_fuga + [target]
X = df_clean.drop(columns=columnas_excluidas)
y = df_clean[target].astype(int)

print("Columnas excluidas:", columnas_excluidas)
print("Características (X):", X.columns.tolist())
print("X:", X.shape, "| y:", y.shape)

abandono,0,1
motivo_baja,,
Atencion,0,277
Competencia,0,291
No aplica,5185,0
Precio,0,247


Columnas excluidas: ['id_cliente', 'motivo_baja', 'abandono']
Características (X): ['edad', 'ingresos_anuales', 'antiguedad_meses', 'saldo_promedio', 'productos_activos', 'canal', 'estado_civil', 'region', 'contacto_preferido', 'incumplimiento']
X: (6000, 10) | y: (6000,)


**Justificación de columnas excluidas**
- **`abandono`**: es la variable objetivo (y), así que se separa de las características.
- **`id_cliente`**: es un identificador único y arbitrario. No describe al cliente, y si se usara el modelo podría memorizar IDs sin aprender patrones generalizables.
- **`motivo_baja`**: **no puede usarse como predictor**. Solo se registra cuando el cliente ya abandonó (vale `"No aplica"` en el 100 % de los no abandonos y tiene un motivo en el 100 % de los abandonos). Es información posterior al evento, es decir, **fuga de información** (*data leakage*). Sirve para análisis descriptivo de causas, pero no para predecir.

Se conservan las demás variables, incluido `incumplimiento`, porque se asume que es un atributo histórico del cliente conocido antes del abandono.

### División experimental - Separe los datos en entrenamiento y prueba con una proporción 80/20, utilizando random_state=42 y preservando la proporción de clases.

In [ ]:
# TODO: aplicar train_test_split con estratificación
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("\nProporción de clases (%):")
display(pd.DataFrame({
    "total": (y.value_counts(normalize=True) * 100).round(2),
    "train": (y_train.value_counts(normalize=True) * 100).round(2),
    "test":  (y_test.value_counts(normalize=True) * 100).round(2),
}))

X_train: (4800, 10) | X_test: (1200, 10)

Proporción de clases (%):


,total,train,test
abandono,,,
0,86.42,86.42,86.42
1,13.58,13.58,13.58


**Justificación:** la división 80/20 con `stratify=y` mantiene la misma proporción de abandono (~13.6 %) en train y en test. Esto es importante por el desbalance de clases: sin estratificar, el conjunto de prueba podría quedar con muy pocos casos positivos. `random_state=42` hace que el resultado sea reproducible. La división se hace **antes** de imputar, codificar o escalar, para evitar fuga del test hacia train.

### Imputación y codificación
- Identifique las características numéricas y categóricas.
- Impute los valores faltantes numéricos con la mediana.
- Impute los valores faltantes categóricos con la moda.
- Codifique las variables categóricas mediante One-Hot Encoding.
- Ajuste los transformadores únicamente con el conjunto de entrenamiento.

In [ ]:
# TODO: construir un ColumnTransformer y los pipelines necesarios
# Recuerde que fit y fit_transform no deben ajustarse sobre el conjunto de prueba.
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_features = X_train.select_dtypes(include="number").columns.tolist()
cat_features = [c for c in X_train.columns if c not in num_features]
print("Numéricas:  ", num_features)
print("Categóricas:", cat_features)

pipeline_num = Pipeline(steps=[
    ("imputador", SimpleImputer(strategy="median")),
])

pipeline_cat = Pipeline(steps=[
    ("imputador", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocesador = ColumnTransformer(transformers=[
    ("num", pipeline_num, num_features),
    ("cat", pipeline_cat, cat_features),
])
print("\nPreprocesador definido (aún sin ajustar).")

Numéricas:   ['edad', 'ingresos_anuales', 'antiguedad_meses', 'saldo_promedio', 'productos_activos']
Categóricas: ['canal', 'estado_civil', 'region', 'contacto_preferido', 'incumplimiento']

Preprocesador definido (aún sin ajustar).


**Justificación:**
- **Mediana** para las numéricas: es robusta frente a atípicos y asimetrías (por ejemplo, en saldo e ingresos), a diferencia de la media.
- **Moda** (`most_frequent`) para las categóricas: asigna la categoría más común sin crear categorías artificiales.
- **One-Hot Encoding**: las categorías son nominales y no tienen orden, así que no se usa codificación ordinal. `handle_unknown="ignore"` evita errores si aparece una categoría nueva en test o en producción.
- Todo se encapsula en un `ColumnTransformer` para que el `fit` se haga **solo con `X_train`**.

### Escalamiento y verificación - Escale las variables numéricas mediante StandardScaler, integrado en el pipeline numérico. Verifique las dimensiones de los datos transformados.

In [ ]:
# TODO: completar el pipeline y transformar X_train y X_test
# Mostrar dimensiones finales y verificar que no existen valores infinitos
# o NaN en las matrices transformadas.

# Escalamiento integrado en el pipeline numérico
pipeline_num = Pipeline(steps=[
    ("imputador", SimpleImputer(strategy="median")),
    ("escalador", StandardScaler()),
])
preprocesador = ColumnTransformer(transformers=[
    ("num", pipeline_num, num_features),
    ("cat", pipeline_cat, cat_features),
])

# fit SOLO con entrenamiento; test solo se transforma
X_train_prep = preprocesador.fit_transform(X_train)
X_test_prep = preprocesador.transform(X_test)

nombres = preprocesador.get_feature_names_out()
X_train_prep = pd.DataFrame(X_train_prep, columns=nombres, index=X_train.index)
X_test_prep = pd.DataFrame(X_test_prep, columns=nombres, index=X_test.index)

print("X_train transformado:", X_train_prep.shape)
print("X_test transformado: ", X_test_prep.shape)
print("Columnas generadas:", len(nombres))

# Medianas aprendidas SOLO con train
medianas = preprocesador.named_transformers_["num"].named_steps["imputador"].statistics_
display(pd.Series(medianas, index=num_features, name="mediana_train").to_frame())

# Verificaciones
for nombre, M in [("train", X_train_prep), ("test", X_test_prep)]:
    n_nan = int(np.isnan(M.values).sum()); n_inf = int(np.isinf(M.values).sum())
    print(f"{nombre}: NaN = {n_nan} | infinitos = {n_inf}")
    assert n_nan == 0 and n_inf == 0
assert X_train_prep.shape[1] == X_test_prep.shape[1]

num_cols_prep = [c for c in nombres if c.startswith("num__")]
print("\nNuméricas escaladas en train (media ≈ 0, desviación ≈ 1):")
display(X_train_prep[num_cols_prep].agg(["mean", "std"]).T.round(3))
display(X_train_prep.head())

X_train transformado: (4800, 19)
X_test transformado:  (1200, 19)
Columnas generadas: 19


,mediana_train
edad,49.000
ingresos_anuales,31721.945
antiguedad_meses,120.000
saldo_promedio,4465.660
productos_activos,2.000


train: NaN = 0 | infinitos = 0
test: NaN = 0 | infinitos = 0

Numéricas escaladas en train (media ≈ 0, desviación ≈ 1):


,mean,std
num__edad,-0.0,1.0
num__ingresos_anuales,-0.0,1.0
num__antiguedad_meses,-0.0,1.0
num__saldo_promedio,0.0,1.0
num__productos_activos,-0.0,1.0


,num__edad,num__ingresos_anuales,num__antiguedad_meses,num__saldo_promedio,num__productos_activos,cat__canal_Digital,cat__canal_Sucursal,cat__canal_Telefono,cat__estado_civil_Casado,cat__estado_civil_Divorciado,cat__estado_civil_Soltero,cat__region_Centro,cat__region_Norte,cat__region_Sur,cat__contacto_preferido_App,cat__contacto_preferido_Email,cat__contacto_preferido_Telefono,cat__incumplimiento_No,cat__incumplimiento_Si
5580,0.106148,-0.661974,1.172966,0.040170,-1.100937,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0
1104,-0.675909,-0.028885,1.216122,0.878913,0.051625,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
2114,0.832343,-1.119227,1.388750,0.288423,0.051625,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
309,-1.402104,0.264007,0.539999,-1.714356,-1.100937,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
1424,-0.843492,-0.393508,1.532606,0.086928,0.051625,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0


**Verificación:** el `StandardScaler` está dentro del pipeline numérico, después de la imputación. En train, las variables numéricas transformadas tienen media ≈ 0 y desviación ≈ 1. En test los valores no son exactamente 0/1, y eso es lo correcto, porque se usan la media y la desviación de train. Las matrices no tienen `NaN` ni infinitos, y train y test tienen el mismo número de columnas, así que están listas para la fase de modelado.

## **Sección E :** (2 PUNTOS) diseño experimental, ética y conclusiones
1. ¿Qué sesgo podría introducir la eliminación indiscriminada de valores atípicos?

2. ¿Por qué se debe ajustar el imputador y el escalador únicamente con los datos de entrenamiento?

3. Explique cómo motivo_baja podría producir fuga de información.

4. Mencione dos riesgos éticos al utilizar información financiera y demográfica.

5. ¿Qué controles propondría para evaluar la calidad, equidad y privacidad del futuro sistema?

6. Concluya si los datos preparados están listos para la siguiente fase y qué limitaciones persisten

### Respuestas – Sección E

**1. Sesgo por eliminación indiscriminada de atípicos.** Muchos "atípicos" son clientes reales con perfiles poco frecuentes: ingresos altos, saldos negativos por sobregiro, clientes muy jóvenes o mayores. Eliminarlos produce un **sesgo de selección**, porque el modelo se entrena con una población más homogénea que la real y falla justamente con esos grupos, que pueden ser los de mayor riesgo de abandono. Además, puede afectar de forma desproporcionada a ciertos segmentos demográficos o económicos. Por eso aquí solo se anularon valores imposibles (edad 250) y códigos centinela (−999999), y se conservaron los extremos plausibles.

**2. ¿Por qué ajustar imputador y escalador solo con train?** Porque el conjunto de prueba simula datos futuros que el modelo no ha visto. Si la mediana, la media o la desviación se calculan con todos los datos, la información de test "se filtra" al preprocesamiento (*data leakage*) y la evaluación resulta optimista. En producción solo se tendrán los parámetros aprendidos en entrenamiento, así que el test debe transformarse con esos mismos parámetros (`fit_transform` en train, `transform` en test).

**3. Fuga de información por `motivo_baja`.** Esta variable se registra **después** de que el cliente abandona: los clientes activos tienen `"No aplica"` y los que se fueron tienen un motivo (Atención, Competencia, Precio). La tabla cruzada muestra que separa perfectamente las dos clases. Si se incluyera, el modelo alcanzaría métricas casi perfectas sin aprender nada útil, porque en el momento de predecir (cliente todavía activo) ese dato no existe. Sería un modelo inútil en producción.

**4. Dos riesgos éticos.**
- **Discriminación o sesgo algorítmico:** variables como `edad`, `estado_civil`, `region` o `ingresos_anuales` pueden llevar a tratar de forma desigual a ciertos grupos (por ejemplo, ofrecer peores condiciones o menos retención a clientes mayores o de ciertas regiones), reproduciendo sesgos históricos.
- **Privacidad y uso indebido de datos personales:** la información financiera y demográfica es sensible. Su filtración, reidentificación o uso para fines distintos al consentido vulnera la privacidad del cliente y normativas como la Ley Orgánica de Protección de Datos Personales del Ecuador.

**5. Controles propuestos.**
- **Calidad:** validaciones automáticas de esquema, rangos y nulos en cada carga (como las de este notebook); métricas adecuadas al desbalance (recall, F1, PR-AUC, matriz de confusión); validación cruzada estratificada y monitoreo de deriva de datos.
- **Equidad:** medir el desempeño y las tasas de falsos positivos y negativos por subgrupos (edad, región, estado civil); aplicar métricas de *fairness* como paridad demográfica o *equal opportunity*; revisar si las variables sensibles son necesarias.
- **Privacidad:** minimización de datos, seudonimización de `id_cliente`, control de accesos y cifrado, consentimiento informado, registro de auditoría y cumplimiento de la normativa de protección de datos.

**6. Conclusión.** Los datos **están listos para la fase de modelado**: se eliminaron 35 duplicados (6035 → 6000 filas), se convirtieron los tipos, se normalizaron las categorías, se trataron los marcadores de ausencia y los valores inválidos, se excluyeron el identificador y la variable con fuga (`motivo_baja`), y se construyó un pipeline de imputación, One-Hot y escalamiento ajustado solo con train, sin `NaN` ni infinitos.

**Limitaciones que persisten:**
- El **desbalance de clases** (~13.6 % de abandono) deberá tratarse en el modelado con ponderación de clases o remuestreo dentro de la validación cruzada, y con métricas adecuadas.
- La **imputación** con mediana y moda es simple: reduce la varianza y no aprovecha relaciones entre variables. No se agregaron indicadores de faltante.
- Los 143 ingresos en 5000 podrían ser un valor censurado o un piso de captura que no se pudo confirmar.
- Hay 987 clientes cuya antigüedad supera los años transcurridos desde sus 18 años. Es una posible inconsistencia entre `antiguedad_meses` y `edad` (o cuentas abiertas siendo menores de edad) que solo se reportó y que debe validarse con el negocio.
- Falta validar con el negocio el significado de los saldos negativos y si `incumplimiento` se conoce antes del abandono.
- Los datos provienen de un solo corte temporal, así que se desconoce su representatividad futura.